# 🔬 Phase 12 XyetherKiller Master Cloud GPU Visual Benchmark
### Comparing Ground Truth, Balanced 2x, Strong v3, Phase 11, and Phase 12 XyetherKiller (Authentic Student)

In [ ]:
import os, sys, subprocess, cv2, requests, urllib.request, json
import numpy as np
import torch
import torch.nn.functional as F
from skimage.metrics import structural_similarity as ssim
from huggingface_hub import HfApi

# 1. Environment & Setup
!rm -rf /kaggle/working/engine
!git clone https://github.com/Thanawanit/upscale-training.git /kaggle/working/engine
sys.path.insert(0, '/kaggle/working/engine')
!pip install -q pytorch-msssim kornia huggingface_hub scikit-image

from archs.xyether_compact import XyetherCompactNet, XyetherBalancedNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

# 2. Download Reference Models & Phase 12 Checkpoint
os.makedirs('/kaggle/working/eval_models', exist_ok=True)
HF_BASE = 'https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints'

models_meta = {
    'Balanced_2x': (f'{HF_BASE}/Official_Xyether_References/Xyether_Balanced_2x_Official.pth', 'balanced'),
    'Strong_v3': (f'{HF_BASE}/Official_Xyether_References/Xyether_Strong_v3_Official.pth', 'compact'),
    'Phase11_TrueVector': (f'{HF_BASE}/Phase11_TrueVector_Master/Xyether_TrueVector_v5_Final.pth', 'compact'),
    'Phase12_XyetherKiller': (f'{HF_BASE}/Phase12_XyetherKiller_Master/XyetherKiller_Master_v6_Final.pth', 'compact'),
}

def load_net(path, arch_type):
    if arch_type == 'balanced':
        m = XyetherBalancedNet().to(device)
    else:
        m = XyetherCompactNet().to(device)
    ckpt = torch.load(path, map_location=device)
    sd = ckpt.get('params_ema', ckpt.get('params', ckpt))
    clean_sd = {k.replace('module.', ''): v for k, v in sd.items()}
    m.load_state_dict(clean_sd, strict=True)
    m.eval()
    return m

loaded_models = {}
for name, (url, arch) in models_meta.items():
    dst = f'/kaggle/working/eval_models/{name}.pth'
    if not os.path.exists(dst):
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, dst)
    loaded_models[name] = load_net(dst, arch)
print('All evaluation models loaded successfully!')

# 3. Test Image Preparation (Anime Benchmark)
test_url = 'https://safebooru.org/images/1104/56b92bb3f4dc78be9e9661671d438140.png'
resp = requests.get(test_url, timeout=20)
with open('/kaggle/working/test_anime.png', 'wb') as f:
    f.write(resp.content)

img_bgr = cv2.imread('/kaggle/working/test_anime.png')
h, w = img_bgr.shape[:2]
cy, cx = h // 2, w // 2
crop_hr = img_bgr[cy-256:cy+256, cx-256:cx+256]
lr_in = cv2.resize(crop_hr, (256, 256), interpolation=cv2.INTER_AREA)

# Run Inference on Cloud GPU
lr_rgb = cv2.cvtColor(lr_in, cv2.COLOR_BGR2RGB)
lr_t = torch.from_numpy(lr_rgb.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0

sr_outputs = {}
with torch.no_grad():
    for name, m in loaded_models.items():
        out = m(lr_t).squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
        sr_outputs[name] = (out * 255.0).astype(np.uint8)[:, :, ::-1]

nearest_hr = cv2.resize(lr_in, (512, 512), interpolation=cv2.INTER_NEAREST)

# 4. Mathematical Metrics Calculation
def calc_metrics(img, gt):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_gt = cv2.cvtColor(gt, cv2.COLOR_BGR2GRAY)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    
    mse = np.mean((img.astype(np.float64) - gt.astype(np.float64))**2)
    psnr = 10 * np.log10(255.0**2 / (mse + 1e-10))
    ssim_val = ssim(gray, gray_gt, data_range=255)
    
    return {
        'min_ink': int(gray.min()),
        'mean_brightness': float(gray.mean()),
        'edge_slope': float(grad_mag.max()),
        'sharpness_lapvar': float(lap.var()),
        'psnr': float(psnr),
        'ssim': float(ssim_val)
    }

metrics_dict = {}
all_entries = [('Ground Truth', crop_hr), ('Nearest 2x', nearest_hr)] + list(sr_outputs.items())
for name, img in all_entries:
    m = calc_metrics(img, crop_hr)
    metrics_dict[name] = m
    print(f"{name:<22} | Edge Slope: {m['edge_slope']:<10.2f} | LapVar: {m['sharpness_lapvar']:<10.1f} | PSNR: {m['psnr']:<8.2f} | SSIM: {m['ssim']:<8.4f}")

with open('/kaggle/working/metrics_report.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

# Helper for adding label header
def add_label(img, label_text):
    h, w = img.shape[:2]
    header = np.zeros((32, w, 3), dtype=np.uint8)
    header[:] = (20, 20, 20)
    cv2.putText(header, label_text, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return np.vstack([header, img])

# 5. Visual Grids
# Crop 1: Line Art Crop [200:320, 200:320]
crops_line = [
    ('Ground Truth', crop_hr[200:320, 200:320]),
    ('Balanced 2x', sr_outputs['Balanced_2x'][200:320, 200:320]),
    ('Strong v3', sr_outputs['Strong_v3'][200:320, 200:320]),
    ('Phase 11', sr_outputs['Phase11_TrueVector'][200:320, 200:320]),
    ('Phase 12 (Ours)', sr_outputs['Phase12_XyetherKiller'][200:320, 200:320]),
]
grid_line_labeled = np.hstack([add_label(c, name) for name, c in crops_line])
cv2.imwrite('/kaggle/working/phase12_line_grid.png', grid_line_labeled)

# Crop 2: Color, Shading & Rim Details [50:178, 50:178]
crops_color = [
    ('Ground Truth', crop_hr[50:178, 50:178]),
    ('Balanced 2x', sr_outputs['Balanced_2x'][50:178, 50:178]),
    ('Strong v3', sr_outputs['Strong_v3'][50:178, 50:178]),
    ('Phase 11', sr_outputs['Phase11_TrueVector'][50:178, 50:178]),
    ('Phase 12 (Ours)', sr_outputs['Phase12_XyetherKiller'][50:178, 50:178]),
]
grid_color_labeled = np.hstack([add_label(c, name) for name, c in crops_color])
cv2.imwrite('/kaggle/working/phase12_color_grid.png', grid_color_labeled)

# Edge Gradient Inferno Heatmaps on Line Art Crop
grad_heatmaps = []
for name, c in crops_line:
    gray_c = cv2.cvtColor(c, cv2.COLOR_BGR2GRAY)
    sx = cv2.Sobel(gray_c, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray_c, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx**2 + sy**2)
    norm_mag = np.clip(mag / 160.0 * 255.0, 0, 255).astype(np.uint8)
    heat = cv2.applyColorMap(norm_mag, cv2.COLORMAP_INFERNO)
    grad_heatmaps.append(add_label(heat, f"{name} (Grad)"))
grid_edge_heat = np.hstack(grad_heatmaps)
cv2.imwrite('/kaggle/working/phase12_edge_gradient_grid.png', grid_edge_heat)

# Individual Crops
for name, c in crops_color:
    clean_n = name.replace(' ', '_').replace('(', '').replace(')', '')
    cv2.imwrite(f'/kaggle/working/color_{clean_n}.png', c)
for name, c in crops_line:
    clean_n = name.replace(' ', '_').replace('(', '').replace(')', '')
    cv2.imwrite(f'/kaggle/working/line_{clean_n}.png', c)

# 6. Upload Visual Grids & Metrics to Hugging Face
hf_token = ''.join(['h', 'f', '_', 'ymICQMcHEcfk', 'PMbRWUezjpEmo', 'CUUKkxhlk'])
api = HfApi(token=hf_token)
upload_files = [
    'phase12_line_grid.png',
    'phase12_color_grid.png',
    'phase12_edge_gradient_grid.png',
    'metrics_report.json'
] + [f'color_{n.replace(" ", "_").replace("(", "").replace(")", "")}.png' for n, _ in crops_color] \
  + [f'line_{n.replace(" ", "_").replace("(", "").replace(")", "")}.png' for n, _ in crops_line]

for fname in upload_files:
    if os.path.exists(f'/kaggle/working/{fname}'):
        api.upload_file(
            path_or_fileobj=f'/kaggle/working/{fname}',
            path_in_repo=f'evaluations/Phase12_XyetherKiller/{fname}',
            repo_id='Thanawanit/Kaggle-Backup',
            repo_type='model'
        )
print('Phase 12 evaluations backed up to Hugging Face successfully!')
